# Lesson 4.1: Extracting Toponyms in Texts

**🎬 Video:** [Lesson 4.1: Extracting Toponyms in Texts](#)

## Overview

Up till now, we have not done anything with the data that is particularly complex. We have found major keywords and trend lines in posting. We don't know that much more about the data yet. In order to do this, we have to use more complex language models to get a better sense of the "about-ness" of the text. In particular, we are interested in *what* locations JMU students talk about and *how* they talk about them as compared to other students. To investigate this we will use a machine learning technique called Named Entity Recognition. In this technique, a computer not only identifies words in a text, but can differentiate what role they play. For our purposes, we are looking at how NER models can be used to extract locations.

This lesson will cover three sections:

- Prepping the data
- Running NER
- Visualizing the data


## Introduction

In this lesson we will learn how to extract place names from text using literal known place names. This is a relatively simple procedure, but we will soon figure out why it is very limited. The meaning of words in human language is always embedded in context. Simply, searching for places by name will not guarantee great results. Consider the following two uses of the word **Washington**: 

- I visited **Washington** last summer and saw the Capitol building.
- **Washington** led the Continental Army during the Revolutionary War.

In the first sentence, "Washington" refers to Washington D.C. (a place). In the second sentence, "Washington" refers to George Washington (a person, not a place). Humans can easily tell the difference, but computers need help figuring this out.

One approach would be to simply make a list of all possible locations and search for that literal location in the text. This would result in a lot of **false positives**, cases where the computer thinks it is true but it is actually false. Therefore linguists started teaching computers the rules of grammar so they could understand **parts of speech**. By knowing *how* a word is being used, computers get more accurate in predicting what the word means in that context. 

By having a basic grammar it is easier to figure out what a word means:

- The weather in London was cold and foggy.
- Jack London wrote *Call of the Wild*.

Here, a computer can figure out that the first "London" is a place (because we say "in London"), while "Jack London" is a person's name because it is performing the verb "write", something cities generally don't do.

Over time, more sophisticated models have developed to help computers understand how language is being used. This process of finding locations, people, organizations, and other important information in text is called Named Entity Recognition (NER). In [Lesson 4.2](./lesson_4_2_using_ner.ipynb) we will see that this is very useful because it helps us understand what a text is really about, beyond just counting words.

---

## 📖 1 Follow Along — Load Libraries and Data

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

Most Python scripts follow the same three-step opening pattern. Recognizing it will help you read unfamiliar code much more quickly:

| Step | What it does | Code pattern |
|---|---|---|
| **Load libraries** | Import the tools you need | `import pandas as pd` |
| **Load data** | Read the dataset into memory | `pd.read_pickle(...)` |
| **Verify data** | Confirm the data looks right before processing | `df.sample(...)` |

This lesson uses `nltk` — Natural Language Toolkit. This is a broad collection of NLP utilities. In this lesson we use its sentence tokenizer to split post text into individual sentences before running entity recognition.

In [ ]:
# ── Step 1: Load libraries ────────────────────────────────────────────────────
# All imports go at the top so dependencies are visible at a glance.
import pandas as pd   # data tables
import nltk           # sentence tokenizer


# ── Step 2: Load data ─────────────────────────────────────────────────────────
# Read the cleaned Reddit DataFrame from the pickle file produced in Lesson 3.
# Pickle preserves column types (category, datetime, etc.) set in that lesson.
df_reddit = pd.read_pickle("../data/JMU/JMU_raw_cleaned.pickle")

# ── Step 3: Verify data ───────────────────────────────────────────────────────
# Always inspect the data before processing.
# .sample() shows random rows rather than just the first ones (.head()),
# which helps catch patterns that only appear further into the dataset.
# random_state=43 makes the sample repeatable — the same random rows will always appear.
df_reddit.sample(n=5, random_state=43)

> 📊 **Output:** The table shows five randomly selected rows from the Reddit dataset. Each row is one post, with columns for `date`, `score`, and `text`. Verify that the data loaded correctly before moving on.

---

## 📖 2 Follow Along — Split `text` into sentences

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

To make our lives a bit easier, we will first split each post into individual sentences. There are two reasons for this:
1. Tokenizers are quicker when they work on short text.
2. We are looking at the emotions around a place-per-sentence. 

The cell below does two things in sequence:

- **`.apply(nltk.sent_tokenize)`** — runs the sentence splitter on every row in the `text` column. Each post becomes a *list* of sentences stored in a new column called `sentences`.
- **`.explode('sentences')`** — takes those lists and gives each sentence its own row. The name sounds dramatic, but all it does is "unpack" the list. A post with 5 sentences becomes 5 rows, each sharing the same metadata (date, score, etc.).

Run the cell to see what the result looks like.

In [ ]:
# Step 1: Split each text into individual sentences using NLTK
df_with_sentence_lists = df_reddit.assign(sentences=df_reddit["text"].apply(nltk.sent_tokenize))

# Step 2: Create a new row for each sentence (explode the lists)
# dropna removes any rows where the sentence is NaN (posts with no text produce empty lists)
df_reddit_sentences = df_with_sentence_lists.explode("sentences").dropna(subset=["sentences"])

# Display a random sample of 5 sentences to see the results
df_reddit_sentences.sample(n=5, random_state=36)


> 📊 **Output:** Each Reddit post has been split into individual sentences, and each sentence now occupies its own row. Notice that `date`, `score`, and `text` all repeat — that data is copied to every sentence that came from the same post. Here is a real example from the dataset where a single post with three sentences becomes three rows:
>
> | date | score | text | sentences |
> |---|---|---|---|
> | 2021-02-04 | 7 | It's been this empty for almost a year now. I'm one of the few people who is there every day. Eerily quiet... | It's been this empty for almost a year now. |
> | 2021-02-04 | 7 | It's been this empty for almost a year now. I'm one of the few people who is there every day. Eerily quiet... | I'm one of the few people who is there every day. |
> | 2021-02-04 | 7 | It's been this empty for almost a year now. I'm one of the few people who is there every day. Eerily quiet... | Eerily quiet... |
>
> The `text` column is now redundant — it's just the `sentences` rows pasted back together. That's why the next step drops it.
> Keep in mind that Python did this for thousands of sentences in a split second. Imagine doing this by hand in a Word doc!

### 2.1 Drop `text`

Since we are essentially copying `text` over and over again it's a good practice to `.drop` it. We already have that information in the new sentences column.

In [ ]:
df_reddit_sentences = df_reddit_sentences.drop(columns=["text"])
df_reddit_sentences.head(5)

> 📊 **Output:** The table is now more compact — only `date`, `score`, and `sentences` remain. Removing columns that are no longer needed is a standard practice in data work. It keeps the DataFrame readable, reduces memory usage, and makes it harder to accidentally use a column you didn't mean to.

---

## 📖 3 Follow Along — Filtering by List

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

The most basic way to figure out if the texts contain places is to go through each text and filter it with a list of known place names. For example, we can create a list of Virginia places:

- 'Richmond'
- 'Harrisonburg'
- 'Lynchburg'
- 'Roanoke'
- 'Charlottesville'


We can then use our `str.contains()` method to filter out any texts that contain the words above.

In [ ]:

# Define the Virginia cities we want to search for
virginia_cities = ["Richmond", "Harrisonburg", "Lynchburg", "Roanoke", "Charlottesville"]

# Filter sentences that contain any of these city names
# The '|' symbol means "OR" - so we're looking for sentences with ANY of these cities
# na=False treats NaN values as non-matching (some posts may have no text)
df_reddit_words = df_reddit_sentences[
    df_reddit_sentences["sentences"].str.contains("|".join(virginia_cities), na=False)
]

# These three examples illustrate what list-based search misses.

# Example 1: "Hburg" — an abbreviation for Harrisonburg that appears in the full sentence set
# but would never be caught by our list, which only contains "Harrisonburg"
ex_hburg = df_reddit_sentences[
    df_reddit_sentences["sentences"].str.contains("Hburg", na=False, case=False)
    & ~df_reddit_sentences["sentences"].str.contains("Harrisonburg", na=False)
].head(1)[["sentences"]]

# Example 2: Harrisonburg + NOVA in the same sentence — we caught the sentence because it has
# "Harrisonburg", but "NOVA" (Northern Virginia) is a place we never thought to include
ex_nova = df_reddit_words[
    df_reddit_words["sentences"].str.contains(r"\bNOVA\b", na=False, regex=True)
].head(1)[["sentences"]]

# Example 3: Harrisonburg + Weyers Cave in the same sentence — we caught the sentence
# because it has "Harrisonburg", but "Weyers Cave" is a place we never thought to include
ex_weyers_cave = df_reddit_words[
    df_reddit_words["sentences"].str.contains("Weyers Cave", na=False)
].head(1)[["sentences"]]

(
    pd.concat([ex_hburg, ex_nova, ex_weyers_cave])
    .reset_index(drop=True)
    .style.set_properties(**{"text-align": "left", "white-space": "normal"})
)


> 📊 **Output:** These three sentences expose the core weakness of searching by a fixed word list. Since you do not know what the data actually contains, you can only make an educated guess as to the type of locations that might occur. This a problem on an **unknown unknown**.
>
> - The first sentence uses "Hburg" — a casual abbreviation for Harrisonburg that students use in everyday writing. Our list only contains "Harrisonburg," so any post that says "Hburg" is invisible to us, no matter how relevant.
> - The second sentence was caught by our filter (it contains "Harrisonburg"), but it also mentions NOVA. Not only is it not on the list, we had not considered adding regions as possible locations.
> - The third sentence was also caught because of "Harrisonburg," but it mentions Weyers Cave, a small town just 15 minutes away that never made it onto our list.
>
> The deeper issue is that you cannot search for what you do not know exists. A list only finds what you already thought to include. Named Entity Recognition, covered in the next section, sidesteps this entirely: rather than matching against a predefined set of strings, it reads the grammar of each sentence and identifies place names on its own. This way it can retrieve locations identified through abbreviations, nicknames, and even places you had never even considered.


### 3.1 Save Sentences for the Next Lesson

The next lesson will run Named Entity Recognition across all of these sentences. To avoid repeating the sentence-splitting work, we save `df_reddit_sentences` as a pickle file so lesson 4.2 can load it directly.

In [ ]:
df_reddit_sentences.to_pickle("../data/JMU/JMU_sentences.pickle")
print(f"✅ Saved {len(df_reddit_sentences):,} rows to JMU_sentences.pickle")

---

## Lesson Summary

**Section 1 — Load Libraries and Data**
- `pd.read_pickle('file.pickle')` — loads a saved DataFrame from a pickle file, preserving column types
- `df.sample(n)` — displays a random selection of rows for a quick sense of the data

**Section 2 — Split `text` into Sentences**
- `df.assign(sentences=...)` — adds a new column by applying a function to an existing one; here used to split post text into individual sentences
- `df.drop(columns=[...])` — removes columns that are no longer needed after the transformation
- `df.explode('sentences', ignore_index=True)` — turns a column of lists into individual rows, one sentence per row

**Section 3 — Filtering by List**
- `df['col'].str.contains(pattern)` — returns a boolean mask identifying rows that match a string pattern or keyword list
- `pd.concat([df_a, df_b], ignore_index=True)` — combines two DataFrames vertically into one
- `df.to_pickle('file.pickle')` — saves a DataFrame to disk so another notebook can load it without repeating the work

➡️ **Next:** [Lesson 4.2 — Named Entity Recognition](lesson_4_2_using_ner.ipynb)